# 🌲 HUẤN LUYỆN CÁC BIẾN THỂ QPSO (QUANTUM-INSPIRED PSO) - FOREST COVERTYPE

Notebook này được thiết lập hoàn chỉnh để bạn chạy huấn luyện các biến thể lượng tử QPSO trên tập dữ liệu **Forest Covertype** (30,000 mẫu phân tầng, 54 đặc trưng, 7 lớp thực vật rừng):

- **Không gian 3D:** $n_{neurons} \in [16, 256]$, $\log_{10}(lr) \in [-4.0, -1.0]$, $\log_{10}(\alpha) \in [-5.0, -1.0]$
- **Hàm mục tiêu:** 5-Fold Stratified Cross-Validation Accuracy trên 24,000 mẫu Train.

## 📦 1. Khai báo Thư viện và Cấu hình Môi trường


In [1]:
import os
import sys
import time
import math
import warnings
warnings.filterwarnings('ignore')

# Cấu hình mã hóa UTF-8 trên Windows để in log không lỗi font
if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.size'] = 11

RANDOM_SEED = 42
print('✅ Khai báo thư viện thành công!')
print(f'   NumPy  : {np.__version__}')
print(f'   Pandas : {pd.__version__}')


✅ Khai báo thư viện thành công!
   NumPy  : 2.4.6
   Pandas : 3.0.3


## 📊 2. Tải và Tiền Xử Lý Dữ Liệu Huấn Luyện (Forest Covertype Data)
Quy trình lấy mẫu phân tầng 30,000 mẫu (24,000 mẫu Train / 6,000 mẫu Test) với 54 đặc trưng chuẩn hóa `StandardScaler`.

In [2]:
CONTINUOUS_COLS = [
    'Elevation', 'Aspect', 'Slope',
    'Horizontal_Distance_To_Hydrology', 'Vertical_Distance_To_Hydrology',
    'Horizontal_Distance_To_Roadways',
    'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm',
    'Horizontal_Distance_To_Fire_Points'
]
WILDERNESS_COLS = [f'Wilderness_Area_{i}' for i in range(1, 5)]
SOIL_COLS = [f'Soil_Type_{i}' for i in range(1, 41)]
ALL_FEATURE_COLS = CONTINUOUS_COLS + WILDERNESS_COLS + SOIL_COLS
TARGET_COL = 'Cover_Type'
COL_NAMES = ALL_FEATURE_COLS + [TARGET_COL]

data_path = '../data/raw/covertype/covtype.data'
if not os.path.exists(data_path):
    data_path = 'data/raw/covertype/covtype.data'
if not os.path.exists(data_path):
    data_path = '../../covertype/data/raw/covertype/covtype.data'

df = pd.read_csv(data_path, header=None, names=COL_NAMES)

# Lấy mẫu phân tầng 30,000 dòng
SAMPLE_SIZE = 30000
working_df, _ = train_test_split(df, train_size=SAMPLE_SIZE, stratify=df[TARGET_COL], random_state=RANDOM_SEED)

X = working_df[ALL_FEATURE_COLS].values
y = (working_df[TARGET_COL].values - 1).astype(int)  # Chuyển nhãn 1-7 về 0-6

# Tách tập Train (80%) và Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

# Chuẩn hóa dữ liệu
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('=' * 65)
print('        DỮ LIỆU ĐÃ SẴN SÀNG CHO TIẾN TRÌNH HUẤN LUYỆN')
print('=' * 65)
print(f'  Tập huấn luyện (Train) : {X_train_sc.shape[0]:,} mẫu | {X_train_sc.shape[1]} đặc trưng')
print(f'  Tập kiểm thử   (Test)  : {X_test_sc.shape[0]:,} mẫu | 7 lớp đa mục tiêu')
print('=' * 65)


        DỮ LIỆU ĐÃ SẴN SÀNG CHO TIẾN TRÌNH HUẤN LUYỆN
  Tập huấn luyện (Train) : 24,000 mẫu | 54 đặc trưng
  Tập kiểm thử   (Test)  : 6,000 mẫu | 7 lớp đa mục tiêu


## 🔧 3. Thiết Lập Không Gian Tìm Kiếm 3D và Hàm Fitness
- **Không gian 3D:** `n_neurons` $\in [16, 256]$, $\log_{10}(lr) \in [-4.0, -1.0]$, $\log_{10}(\alpha) \in [-5.0, -1.0]$.

In [3]:
# --- Không gian 3D ---
LOW  = np.array([16,  -4.0, -5.0])
HIGH = np.array([256, -1.0, -1.0])
DIM  = 3

def decode_position(x):
    n_neurons = int(round(np.clip(x[0], LOW[0], HIGH[0])))
    lr        = float(10 ** np.clip(x[1], LOW[1], HIGH[1]))
    alpha     = float(10 ** np.clip(x[2], LOW[2], HIGH[2]))
    return n_neurons, lr, alpha

def fitness_single(x, X_tr, y_tr, cv_obj, seed):
    n_neurons, lr, alpha = decode_position(x)
    model = MLPClassifier(
        hidden_layer_sizes=(n_neurons, n_neurons),
        activation='relu',
        learning_rate_init=lr,
        alpha=alpha,
        max_iter=500,
        random_state=seed,
        early_stopping=True,
        n_iter_no_change=15
    )
    scores = cross_val_score(model, X_tr, y_tr, cv=cv_obj, scoring='accuracy', n_jobs=-1)
    return float(scores.mean())

print('✅ Đã thiết lập Không gian tìm kiếm 3D!')


✅ Đã thiết lập Không gian tìm kiếm 3D & 5D!


---
## 📈 4. Đối Sánh Tiến Trình Hội Tụ Của Các Biến Thể Lượng Tử
Sau khi chạy xong các thuật toán, bạn có thể chạy cell bên dưới để vẽ biểu đồ so sánh đường cong hội tụ (Convergence Curve) trực quan giữa các biến thể.

In [ ]:
# Đối sánh đường cong hội tụ các biến thể QPSO đã hoàn thành
checkpoints_to_plot = {
    'QPSO': '../results/checkpoints/qpso.pkl',
    'Q-LDPSO': '../results/checkpoints/q_ldpso.pkl',
    'Q-MBPSO': '../results/checkpoints/q_mbpso.pkl',
    'QPSO-Nonlinear': '../results/checkpoints/qpso_nonlinear.pkl',
    'QPSO-Adaptive': '../results/checkpoints/qpso_adaptive.pkl',
    'QPSO-PerParticle': '../results/checkpoints/qpso_per_particle.pkl',
    'OQ-PSO': '../results/checkpoints/oq_pso.pkl'
}

plt.figure(figsize=(11, 6))
loaded_count = 0

for name, path in checkpoints_to_plot.items():
    if os.path.exists(path):
        data = joblib.load(path)
        conv = data.get('convergence', [])
        if len(conv) > 0:
            plt.plot(range(1, len(conv) + 1), conv, marker='o', markersize=4, label=f"{name} (CV: {data['cv_score']:.4f})")
            loaded_count += 1

if loaded_count > 0:
    plt.xlabel('Số vòng lặp (Iteration)', fontsize=12)
    plt.ylabel('Điểm thích nghi CV Accuracy', fontsize=12)
    plt.title('So sánh tiến trình hội tụ các biến thể QPSO trên Forest Covertype', fontsize=13, fontweight='bold')
    plt.legend(loc='lower right', frameon=True)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('ℹ️ Chưa có checkpoint nào được hoàn thành để vẽ biểu đồ. Hãy chạy các cell huấn luyện bên trên!')
